# Setup

In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

In [2]:
import os
import sys
from pprint import pprint

# so that mllm_shap can be imported without installing the package
sys.path.insert(0, os.path.abspath("../mllm_shap/src"))

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"
os.environ["LOG_LEVEL"] = "INFO"

In [3]:
import numpy as np
import pandas as pd
import torch

np.random.seed(42)

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"Using device: {device}")

Using device: mps


In [4]:
from mllm_shap.connectors import LiquidAudio, ModelConfig
from mllm_shap.connectors.enums import ModelHistoryTrackingMode, Role, SystemRolesSetup
from mllm_shap.connectors.filters import ExcludePunctuationTokensFilter
from mllm_shap.shap import Explainer, McShapExplainer
from mllm_shap.shap.embeddings import MeanReducer
from mllm_shap.shap.enums import Mode
from mllm_shap.shap.normalizers import PowerShiftNormalizer
from mllm_shap.shap.similarity import TfIdfCosineSimilarity
from mllm_shap.utils.jupyter import display_shap_colors_df

# Usage

Define LiquidAudio model (this call loads it up to the memory!).

Create compact explainer that will make initial call and then explain it using Shapley Values approximated using Monte Carlo. Set minimal number of samples for demo purpose, that is only first-order omission ones (where only one token at the time is hidden) and empty sample (if it is not "globally" empty, that is when expandability is not performed on all tokens).

PowerNormalizer first shifts all shapley values by subtracting their minimum (so new minimal value will be 0.0), then raises them to power of 2.0 and normalizes so they sum to 1.0.

In [5]:
model = LiquidAudio(
    device=device, history_tracking_mode=ModelHistoryTrackingMode.TEXT
)  # track and generate only text history
shap = McShapExplainer(
    num_samples=-1,
    mode=Mode.CONTEXTUAL,  # use contextual embeddings, default
    # use mean pooling to reduce token embeddings to single embedding per audio, default
    embedding_reducer=MeanReducer(),
    similarity_measure=TfIdfCosineSimilarity(),  # use TF-IDF weighted cosine similarity to compare embeddings
    normalizer=PowerShiftNormalizer(power=2.0),  # use power-shift normalization with power of 2.0
)
explainer = Explainer(model=model, shap_explainer=shap)

W1108 11:53:41.378000 35268 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Create new chat that treats Assistant messages as system, that is will ignore them for shapley values calculation - they will be feed to each prompt as system messages. This significantly reduces number of requests needed for multi turn expandability, yet might not be possible due to business requirements. 

To further reduce number of calls we exclude punctuation tokens.

In [6]:
chat = model.get_new_chat(
    system_roles_setup=SystemRolesSetup.SYSTEM_ASSISTANT,
    token_filter=ExcludePunctuationTokensFilter(),  # exclude punctuation tokens from shapley values calculation
)

chat.new_turn(Role.SYSTEM)
chat.add_text("You are a helpful assistant that answers questions briefly.")
chat.end_turn()

chat.new_turn(Role.USER)
chat.add_text("Who are you?")
chat.end_turn()

Let's have a look at chat representation:

In [7]:
chat.get_conversation()

[[ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='<|im_start|>, system, \n, You,  are,  a,  helpful,  assistant,  that,  answers,  questions,  briefly...', shap_values=None)],
 [ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='<|im_start|>, user, \n, Who,  are,  you, ?, <|im_end|>, \n...', shap_values=None)]]

Representation is a list of list of ConversationEntry - so it can be accessed as  chat.get_conversation()[turn_number][message_number].{field}

Let's calculate shapley values for current conversation.

Verbose=True allows us to access history object (descried later). Generation kwargs allows to customize model interference - here we limit it to 64 tokens and change text_temperature from default 0.0 to 0.2. 

In [8]:
generation_kwargs = {"max_new_tokens": 64, "model_config": ModelConfig(text_temperature=0.2)}

result = explainer(
    chat=chat,
    verbose=True,
    generation_kwargs=generation_kwargs,
    progress_bar=True,  # show progress bar during generation, default
)

2025-11-08 11:53:46,098 - mllm_shap.shap.compact - INFO - Generating full response from the model...
2025-11-08 11:53:47,609 - mllm_shap.shap.base.explainer - INFO - Number of tokens for explainability: 3 (up to 7 additional calls)


Calculating SHAP values:   0%|          | 0/4 [00:00<?, ?it/s]

2025-11-08 11:53:52,226 - mllm_shap.shap.base.explainer - INFO - Deduplicated 0/4 masks using existing cache.


Result has now following fields available:

- full_chat - chat with base response (generated based on whole entry) with set cache and calculated shapley values
- source_chat - original chat feed to the explainer
- history - history of all chats used

Cache object stores actual shapley values as well as calculated embeddings and masks. They will be reused in next call regardless to the method, so for monte-carlo it is just larger sample, for precise it means some results might get excluded.

Let's first analyze history - it is a list of size equivalent to number of calls made for calculations + 1 (first entry, for base calculations, always None) - in this case, 4 (3 for base one-versus-all and one for empty call). Each entry is a tuple of following values:

- mask for that entry
- mash hash
- source chat with masked entry or None if corresponding mask was available in cache
- model response object

or None - when either corresponding mask was extracted from cache or it has risen an AllTextTokensFilteredOutError error.

Let's see all chats that were taken into account:

In [9]:
[c[2].decode_text() if c is not None else None for c in result.history]

['<|startoftext|><|im_start|>system\nYou are a helpful assistant that answers questions briefly.<|im_end|>\n<|im_start|>user\n?<|im_end|>\n',
 '<|startoftext|><|im_start|>system\nYou are a helpful assistant that answers questions briefly.<|im_end|>\n<|im_start|>user\n are you?<|im_end|>\n',
 '<|startoftext|><|im_start|>system\nYou are a helpful assistant that answers questions briefly.<|im_end|>\n<|im_start|>user\nWho you?<|im_end|>\n',
 '<|startoftext|><|im_start|>system\nYou are a helpful assistant that answers questions briefly.<|im_end|>\n<|im_start|>user\nWho are?<|im_end|>\n']

We can see that "?" was never removed, as it is present even in the empty mask. For rest, we can see that all system tokens are always present and only user tokens get masked. out between each calls.

Let's now analyze calculated shapley values.

In [10]:
explained_chat = result.full_chat

explained_chat_conversation = explained_chat.get_conversation()
pprint(explained_chat_conversation)

[[ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='<|im_start|>, system, \n, You,  are,  a,  helpful,  assistant,  that,  answers,  questions,  briefly...', shap_values=[nan, nan, ..., nan, nan])],
 [ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='<|im_start|>, user, \n, Who,  are,  you, ?, <|im_end|>, \n...', shap_values=[nan, nan, ..., nan, nan])],
 [ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='<|im_start|>, assistant, \n, I,  am,  an,  AI,  language,  model,  created,  by,  Open, AI, ,,  desi...', shap_values=[nan, nan, ..., nan, nan])]]


Model was set to return just text tokens, so chat history has only text tokens. We can see that in the json representation inside ConversationEntry shap_values field is now populated. Nan values indicated that this token wasn't taken into calculation scope. As expected, we have 3 not-nan tokens. Let's see them.

In [11]:
user_entry = explained_chat_conversation[1][0]

display_shap_colors_df(
    pd.DataFrame(
        list(zip(user_entry.content, user_entry.shap_values, user_entry.roles)),
        columns=["Token", "Shapley Value", "Role"],
    )
)

,Token,Shapley Value,Role
0,<|im_start|>,nan,2
1,user,nan,2
2,,nan,2
3,Who,0.317889,0
4,are,0.000000,0
5,you,0.682111,0
6,?,nan,0
7,<|im_end|>,nan,2
8,,nan,2


Let's create another turn to see how input significance will change:

In [12]:
explained_chat.new_turn(Role.USER)
explained_chat.add_text("Can you repeat?")
explained_chat.end_turn()

And again, let's explain it:

In [13]:
result = explainer(chat=explained_chat, verbose=True, generation_kwargs=generation_kwargs)

2025-11-08 11:53:52,576 - mllm_shap.shap.compact - INFO - Generating full response from the model...
2025-11-08 11:53:54,193 - mllm_shap.shap.base.explainer - INFO - Number of tokens for explainability: 6 (up to 63 additional calls)


Calculating SHAP values:   0%|          | 0/7 [00:00<?, ?it/s]

2025-11-08 11:54:05,018 - mllm_shap.shap.base.explainer - INFO - Deduplicated 0/7 masks using existing cache.


In [14]:
[c[2].decode_text() if c is not None else None for c in result.history]

['<|startoftext|><|im_start|>system\nYou are a helpful assistant that answers questions briefly.<|im_end|>\n<|im_start|>user\n?<|im_end|>\n<|im_start|>assistant\nI am an AI language model created by OpenAI, designed to assist with a wide range of questions and tasks. How can I help you today?<|im_end|><|im_end|>\n<|im_start|>user\n?<|im_end|>\n',
 '<|startoftext|><|im_start|>system\nYou are a helpful assistant that answers questions briefly.<|im_end|>\n<|im_start|>user\n are you?<|im_end|>\n<|im_start|>assistant\nI am an AI language model created by OpenAI, designed to assist with a wide range of questions and tasks. How can I help you today?<|im_end|><|im_end|>\n<|im_start|>user\nCan you repeat?<|im_end|>\n',
 '<|startoftext|><|im_start|>system\nYou are a helpful assistant that answers questions briefly.<|im_end|>\n<|im_start|>user\nWho you?<|im_end|>\n<|im_start|>assistant\nI am an AI language model created by OpenAI, designed to assist with a wide range of questions and tasks. How c

In [15]:
explained_chat = result.full_chat

explained_chat_conversation = explained_chat.get_conversation()
pprint(explained_chat_conversation)

[[ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='<|im_start|>, system, \n, You,  are,  a,  helpful,  assistant,  that,  answers,  questions,  briefly...', shap_values=[nan, nan, ..., nan, nan])],
 [ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='<|im_start|>, user, \n, Who,  are,  you, ?, <|im_end|>, \n...', shap_values=[nan, nan, ..., nan, nan])],
 [ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='<|im_start|>, assistant, \n, I,  am,  an,  AI,  language,  model,  created,  by,  Open, AI, ,,  desi...', shap_values=[nan, nan, ..., nan, nan])],
 [ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='<|im_start|>, user, \n, Can,  you,  repeat, ?, <|im_end|>, \n...', shap_values=[nan, nan, ..., nan, nan])],
 [ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='<|im_start|>, assistant, \n, I,  am,  an,  AI,  language,  model,  created,  by,  O

In [16]:
dt = []
for i in (1, 3):
    user_entry = explained_chat_conversation[i][0]
    df = pd.DataFrame(
        list(zip(user_entry.content, user_entry.shap_values, user_entry.roles)),
        columns=["Token", "Shapley Value", "Role"],
    )
    df["Turn"] = i
    dt.append(df)

df = pd.concat(dt).reset_index(drop=True)
display_shap_colors_df(df)

,Token,Shapley Value,Role,Turn
0,<|im_start|>,nan,2,1
1,user,nan,2,1
2,,nan,2,1
3,Who,0.231312,0,1
4,are,0.000000,0,1
5,you,0.229981,0,1
6,?,nan,0,1
7,<|im_end|>,nan,2,1
8,,nan,2,1
9,<|im_start|>,nan,2,3
